# 01 — Analisi esplorativa del dataset

**Domanda di questo notebook:** che cosa c'è davvero dentro `anamnesiterapie.txt`,
e che cosa ci obbliga a fare nel resto del progetto?

Ogni sezione misura una cosa e poi dice **che conseguenza ha sulla progettazione**.
Un numero senza la sua conseguenza non serve a niente: qui interessa il secondo,
il primo è solo l'evidenza.

Il file usato è l'**export grezzo** del sistema ospedaliero. Nel progetto esistono
anche varianti già ripulite e strutturate, ma sono state prodotte facendo passare
i dati attraverso un modello linguistico: costruirci sopra la pipeline
significherebbe misurare l'estrazione di qualcun altro invece della nostra.

In [ ]:
import re
import sys
import statistics
from collections import Counter
from pathlib import Path

RADICE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RADICE / "src"))

from data_loading import carica_dataset

record, anomalie = carica_dataset(RADICE / "data" / "raw" / "anamnesiterapie.txt")
print(f"record caricati: {len(record)}")
print(f"anomalie di caricamento: {len(anomalie)}")

## 1. Quanti pazienti, e quanti sono utilizzabili

`encOid` identifica il **ricovero** (*encounter*), non la persona. È l'unico
identificativo disponibile, e nel dataset risulta unico: lo trattiamo come chiave
primaria sapendo che due ricoveri dello stesso paziente sarebbero due record
distinti e noi non possiamo accorgercene.

In [ ]:
identificativi = [r.enc_oid for r in record]
con_dimissione = [r for r in record if r.ha_terapia_dimissione]

print(f"ricoveri totali            {len(record)}")
print(f"identificativi distinti    {len(set(identificativi))}")
print(f"con terapia alla dimissione {len(con_dimissione)}")
print(f"senza                       {len(record) - len(con_dimissione)}")

**Cosa se ne deduce.** La terapia alla dimissione è ciò che il sistema deve
imparare a raccomandare: è quindi la nostra *ground truth*, e solo 857 record ce
l'hanno.

I 143 record senza non vanno buttati, ed è una decisione presa in modo esplicito.
Non possono valutare la raccomandazione, ma possono valutare **tutto il resto**:
l'estrazione delle entità, il collegamento ai codici, la negazione. Buttarli
significherebbe rinunciare al 14% dei dati di test per una capacità che a quei
record non serve.

## 2. Quanto testo c'è per paziente

La lunghezza non è curiosità statistica: decide la finestra di contesto del
modello, il tempo di inferenza e quanto costa una corsa completa.

In [ ]:
def lunghezze(estrattore):
    return [len(estrattore(r) or "") for r in record if estrattore(r)]

campi = {
    "Anamnesi": lambda r: r.testo_anamnesi,
    "Terapia all'ingresso": lambda r: r.testo_terapia_ingresso,
    "Terapia alla dimissione": lambda r: r.testo_terapia_dimissione,
}

print(f"{'campo':26} {'presenti':>9} {'media':>8} {'mediana':>8} {'p95':>7} {'max':>7}")
for nome, f in campi.items():
    v = sorted(lunghezze(f))
    print(f"{nome:26} {len(v):9} {statistics.mean(v):8.0f} "
          f"{v[len(v)//2]:8} {v[int(len(v)*0.95)]:7} {v[-1]:7}")

totali = sorted(sum(len(x.testo or "") for x in r.referti) for r in record)
print(f"\n{'TUTTI I CAMPI INSIEME':26} {len(totali):9} {statistics.mean(totali):8.0f} "
      f"{totali[len(totali)//2]:8} {totali[int(len(totali)*0.95)]:7} {totali[-1]:7}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(totali, bins=60, color="#4C6EF5", edgecolor="white", linewidth=0.4)
ax.axvline(statistics.median(totali), color="#E8590C", linewidth=2,
           label=f"mediana {statistics.median(totali):.0f}")
ax.axvline(totali[int(len(totali)*0.95)], color="#0CA678", linewidth=2, linestyle="--",
           label=f"95° percentile {totali[int(len(totali)*0.95)]}")
ax.set_xlabel("caratteri per ricovero (tutti i campi)")
ax.set_ylabel("numero di ricoveri")
ax.set_title("La distribuzione ha una coda lunga: la media inganna")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Cosa se ne deduce.** La distribuzione è **asimmetrica con una coda lunga**: la
mediana sta intorno ai 2 500 caratteri ma il massimo supera i 18 000. Progettare
sulla media sarebbe un errore — sono i record della coda a rompere le cose.

Tre conseguenze concrete, tutte verificate poi sul campo:

1. Una finestra di contesto di **8 192 token** basta per il 99% dei record. Non è
   una scelta di comodo: è il massimo che 14 GiB di RAM reggono con un modello da
   4 miliardi di parametri.
2. Il tempo di inferenza **non** è proporzionale alla lunghezza del referto. La
   correlazione misurata fra token in ingresso e token in uscita è **r = 0,52**:
   la lunghezza influenza la risposta ma non la determina. Due record su 200
   hanno esaurito trenta minuti di generazione, e non erano i più lunghi.
3. Con una mediana così bassa, la maggior parte dei referti è **breve e telegrafica**.
   È il motivo per cui le regole di prossimità della sezione 4 funzionano: c'è
   poca subordinazione da attraversare.

## 3. Che forma hanno i campi

I tre campi hanno nature diverse, e la differenza determina come vanno estratti.

> **Nota sui dati mostrati da qui in avanti.** Le statistiche di questo notebook
> sono calcolate sul file clinico vero. Gli **esempi di testo** invece no: sono
> costruiti a mano riproducendo fedelmente il formato reale, campo per campo e
> separatore per separatore.
>
> Il file `anamnesiterapie.txt` è pseudonimizzato ma non versionato — `.gitignore`
> lo esclude — e un notebook con gli output salvati finirebbe su GitHub portandosi
> dietro il testo dei referti. Le stesse celle, eseguite da chi ha il file, danno
> gli stessi risultati sui dati veri: cambiare `esempio` con un record vero è una
> riga.

In [ ]:
from data_loading import RecordPaziente, Referto

# Record SINTETICO, inventato riproducendo il formato reale campo per campo.
# Il dataset non e' versionato: il notebook gira anche senza.
# Per lavorare sui dati veri basta sostituire questa riga con:
#     esempio = next(r for r in record if r.ha_terapia_dimissione and r.testo_anamnesi)
esempio = RecordPaziente(enc_oid=0, referti=[
    Referto("Anamnesi", None, None, 'Allergie e intolleranze: Allergie e intolleranze non note Anamnesi Remota: donna di 71 anni, ex fumatrice. Nega intolleranza a mezzi di contrasto. Ipotiroidismo, in trattamento sostitutivo stabile. Nega ipertensione. Familiarita per cardiomiopatia dilatativa (fratello). Non versamento pleurico. Pregresso flutter atriale tipico, ablato nel 2019.'),
    Referto("Terapia medica all'ingresso", None, None, 'Levotiroxina teva: 88 mcg cpr. /die (ore 7) ; Ramipril doc: 2,5 mg cps. rigide /die (ore 21) ; Colecalciferolo abc: 25000 UI gtt. os (ore 9) ogni 14 giorni ;'),
    Referto("Terapia alla Dimissione", None, None, '"Levotiroxina sodica (Levotiroxina teva cpr. 88 mcg): da assumere 88 mcg (ore 7)" "Ramipril (Ramipril doc cps. rigide 2,5 mg): da assumere 2,5 mg (ore 21)" "Colecalciferolo (Colecalciferolo abc gtt. os 25000 UI): da assumere 25000 UI (ore 9), ogni 14 giorni"'),
])

def mostra(titolo, testo, n=280):
    print(f"--- {titolo} ---")
    print((testo or "")[:n].replace("\n", " ") + ("\u2026" if len(testo or "") > n else ""))
    print()

mostra("ANAMNESI (prosa libera)", esempio.testo_anamnesi)
mostra("TERAPIA ALL'INGRESSO (semi-strutturata)", esempio.testo_terapia_ingresso)
mostra("TERAPIA ALLA DIMISSIONE (semi-strutturata, altro formato)", esempio.testo_terapia_dimissione)

**Cosa se ne deduce.** L'anamnesi è **prosa libera** e richiede riconoscimento di
entità. I due campi di terapia sono **semi-strutturati**, con separatori regolari,
e per loro una regex è più affidabile di qualunque modello: è deterministica,
ispezionabile e non inventa.

Ma i due formati **non sono lo stesso formato**, ed è la prossima sezione.

## 4. Espressioni regolari: cosa si può fare senza modelli

Qui le regex non sono un esercizio: sono la pipeline A, la nostra linea di base.
Ognuna di queste è nel codice del progetto.

In [ ]:
# --- regex 1: separare le voci della terapia alla DIMISSIONE ---------------
# Formato reale, virgolette comprese:
#   "Principio attivo (Nome commerciale forma dose): da assumere dose (orario)"
VOCE_DIMISSIONE = re.compile(r'"([^"(]+?)\s*\(([^)]*)\)\s*:\s*([^"]*)"')

testo = esempio.testo_terapia_dimissione
print("TERAPIA ALLA DIMISSIONE — principio attivo, nome commerciale, posologia")
for principio, commerciale, posologia in VOCE_DIMISSIONE.findall(testo)[:6]:
    print(f"  {principio.strip():20} | {commerciale.strip():28} | {posologia.strip()[:30]}")

In [ ]:
# --- regex 2: la terapia all'INGRESSO usa un separatore diverso ------------
# Formato:  Nome commerciale: dose forma /die (orario)
VOCE_INGRESSO = re.compile(r"([^;:]+):\s*([^;]+)")

testo = esempio.testo_terapia_ingresso
print("TERAPIA ALL'INGRESSO — qui il separatore sono i DUE PUNTI, non la parentesi")
for nome, posologia in VOCE_INGRESSO.findall(testo)[:6]:
    print(f"  farmaco: {nome.strip():30} posologia: {posologia.strip()[:44]}")

> **Un difetto reale, trovato proprio da questa differenza.** La scomposizione
> delle menzioni composte gestiva solo il formato di dimissione, che separa con
> la parentesi. Sul formato d'ingresso, che separa con i due punti, **14 farmaci
> su 14 restavano senza codice ATC** in una verifica su due record. Corretta la
> regola, la copertura di quei record è passata da **0% a 100%**.
>
> È il motivo per cui questa sezione esiste: guardare i due formati affiancati
> rende ovvio un difetto che sui numeri aggregati era invisibile.

In [ ]:
# --- regex 3: i marcatori di negazione, contati sul corpus ------------------
# Non tradotti dall'inglese di medspaCy: contati qui dentro, uno per uno.
tutto = "\n".join(x.testo for r in record for x in r.referti if x.testo).lower()

MARCATORI = ["non ", "nega ", "assenza di", "senza ", "negativo per",
             "escluso", "asintomatico per", "non riferit"]
print(f"corpus: {len(tutto):,} caratteri\n")
print("marcatori di NEGAZIONE, per frequenza reale:")
for m in sorted(MARCATORI, key=lambda x: -tutto.count(x)):
    print(f"  {m:20} {tutto.count(m):6}")

In [ ]:
# --- regex 4: la familiarita', che e' un asse a se' ------------------------
FAMILIARITA = re.compile(
    r"familiarit[aà]\W{0,3}(?:positiva|negativa)?\W{0,3}per\b|familiarit[aà]\b"
    r"|anamnesi\s+familiare\b", re.IGNORECASE)

occorrenze = FAMILIARITA.findall(tutto)
print(f"marcatori di familiarita': {len(occorrenze)}")
print("  'familiarita per'      ", tutto.count("familiarita per"))
print("  'familiarita positiva' ", tutto.count("familiarita positiva"))
print("  'familiarita negativa' ", tutto.count("familiarita negativa"))
print("  'anamnesi familiare'   ", tutto.count("anamnesi familiare"))
print()
print("termini di parentela, frequenti ma NON usati come marcatori:")
print(f"  {'termine':12} {'sottostringa':>13} {'parola intera':>14}")
for p in ("padre", "madre", "fratello", "sorella", "zio", "nonno"):
    intera = len(re.findall(rf"\b{p}\b", tutto))
    print(f"  {p:12} {tutto.count(p):13} {intera:14}")

**Cosa se ne deduce, ed è la scoperta più importante del progetto.**

I marcatori di negazione **vanno contati sul corpus, non tradotti**. `denies`,
`no evidence of`, `rule out` di medspaCy in un referto italiano non compaiono
mai. Ma la stessa lista mostra anche i propri buchi: `asintomatico per` e
`non riferit-` esistono nel corpus e **non erano nella lista**, e infatti la
pipeline A sbaglia esattamente su quelle frasi.

La familiarità è un caso a parte. `familiarità per` compare 431 volte, ed è una
domanda **diversa** dalla negazione: *«Familiarità per cardiopatia ischemica
(padre)»* non dice che il paziente ce l'ha né che non ce l'ha — dice che ce l'ha
suo padre. È l'asse *experiencer* di ConText, e nella prima versione dello schema
mancava: la pipeline A rispondeva `affermato`, la pipeline B `negato`, e **nessuna
delle due aveva ragione**.

I termini di parentela sono frequenti ma **non sono marcatori**: nel corpus
compaiono quasi sempre come precisazione fra parentesi *dentro* un ambito già
aperto da `familiarità`. Promuoverli catturerebbe qualche caso in più e ne
romperebbe molti altri.

> **La trappola del conteggio per sottostringa.** Guardate le due colonne di
> `zio`. Contato come sottostringa dà migliaia di occorrenze, perché finisce
> dentro `funzione`, `sostituzione`, `ospedalizzazione`. Contato come parola
> intera crolla a poche decine.
>
> È un errore che ho commesso scrivendo questa cella, e vale la pena lasciarlo
> visibile: un marcatore costruito su quel numero avrebbe aperto ambiti di
> familiarità su mezzo corpus. **Ogni conteggio lessicale va fatto a confine di
> parola** — `\b` — se non si vuole misurare il rumore invece del segnale.

In [ ]:
# --- regex 5: la sezione allergie, e perche' una lista vuota e' ambigua -----
SEZIONE_ALLERGIE = re.compile(r"allerg|intolleran|anafila", re.IGNORECASE)

con_sezione = sum(1 for r in record if SEZIONE_ALLERGIE.search(r.testo_anamnesi or ""))
NON_NOTE = re.compile(r"non note|nessuna allergia|negativ[ao] per allerg|nega allerg",
                      re.IGNORECASE)
dichiarate_assenti = sum(1 for r in record
                         if NON_NOTE.search(r.testo_anamnesi or ""))

print(f"ricoveri la cui anamnesi nomina le allergie:  {con_sezione:4} ({con_sezione/len(record):.1%})")
print(f"  di cui dichiarano esplicitamente 'assenti': {dichiarate_assenti:4}")
print(f"ricoveri che non ne parlano affatto:          {len(record)-con_sezione:4} "
      f"({(len(record)-con_sezione)/len(record):.1%})")

**Cosa se ne deduce.** Una lista di allergie vuota è **ambigua**: può voler dire
«il clinico ha verificato e non ce ne sono» oppure «il referto non ne parla».
Sono cose diverse, e per un filtro di sicurezza la differenza è tutto.

È il motivo per cui lo schema ha un campo `stato_sezione_allergie` **separato**
dalla lista: `negato` significa verificato, `ignoto` significa non sappiamo.

Il 41% dei referti non nomina affatto le allergie, e di quelli che le nominano
**due terzi lo fanno per dire che non ce ne sono**: è una sezione che il clinico
compila per negare, non per affermare.

Questa misura ha poi permesso di scoprire il difetto più grave della pipeline B:
su 24 record dei 198 elaborati, il modello ha estratto allergie da referti che
la parola «allergia» non la contengono affatto — e in un caso le dieci «allergie»
erano **la lista dei farmaci che il paziente assume**.

## 5. Quante entità ci aspettiamo per record

Ultimo numero, e serve a dimensionare i vincoli sull'uscita del modello.

In [ ]:
import json
percorso_a = RADICE / "data" / "processed" / "pipeline_a"
if percorso_a.exists():
    stati = [json.loads(p.read_text()) for p in percorso_a.glob("*.json")
             if not p.stem.startswith("_")]
    cond = sorted(len(s["condizioni"]) for s in stati)
    farm = sorted(len(s["farmaci"]) for s in stati)
    print(f"pipeline A su {len(stati)} record")
    print(f"  condizioni per record: mediana {cond[len(cond)//2]}, "
          f"95° pct {cond[int(len(cond)*.95)]}, max {cond[-1]}")
    print(f"  farmaci per record:    mediana {farm[len(farm)//2]}, "
          f"95° pct {farm[int(len(farm)*.95)]}, max {farm[-1]}")
else:
    print("pipeline A non ancora eseguita: `python3 src/extract_a.py`")

**Cosa se ne deduce.** Con una mediana di poche condizioni per record e un
massimo di poche decine, un tetto di **60 elementi per array** nello schema di
uscita è largamente generoso per i record sani e taglia solo quelli patologici.

Non è una preferenza stilistica: `maxItems` lo applica **il decodificatore
vincolato**, non il prompt, quindi è un vincolo che il modello non può ignorare —
ed è l'unico rimedio strutturale alla sovra-estrazione che ha fatto fallire due
record su 200.

---

## Riepilogo: cosa questa analisi ha deciso

| misura | conseguenza progettuale |
|---|---|
| 1 000 ricoveri, 857 con terapia alla dimissione | i 143 senza restano, per valutare estrazione e codifica |
| mediana ~2 500 caratteri, coda fino a 18 000 | contesto a 8 192 token; progettare sulla coda, non sulla media |
| due formati di terapia con separatori diversi | la scomposizione deve gestirli entrambi (difetto reale trovato così) |
| `non` 2 665, `nega` 522, `assenza di` 370 | marcatori contati sul corpus, non tradotti dall'inglese |
| `asintomatico per`, `non riferit-` presenti ma fuori lista | limite noto e misurato della pipeline A |
| `familiarità per` 431, parentele escluse | asse *experiencer* separato dallo stato clinico |
| il 52,5% dei referti non nomina le allergie | `stato_sezione_allergie` separato dalla lista |
| poche decine di entità per record | `maxItems: 60` come vincolo del decodificatore |